# Irrigation Training - v2.19b-TD3 (CORRECTED: anti-collapse exploration)

**This is the corrected re-run of v2.19.** Architecture, env, observation, reward, and the V211 LayerNorm VDN critic are **byte-identical** to v2.19 (deterministic `_TD3SharedActor`, marker 2.19; `runner.py` dispatches it unchanged via marker>=2.185 + no log_std -> `TD3.load`). Only the **training dynamics** change.

## Why v2.19 collapsed (re-derived from the committed eval files)

v2.19 removed the SAC entropy term to unpin the actor from the 6 mm centre so it could reach 0 mm on wet days. It worked too well: the deterministic actor collapsed to ~0 mm in **every** scenario.
- eval mean-reward flat at **~-6** for all 250k steps (never improved; SAC v2.18 sat near 0);
- twin-Q critic **diverged**: q_pred_mean -21 -> -57 -> -105 -> **-145** while realised return was only ~-4;
- trained policy applied <0.5 mm on **69% (dry) / 93% (mod) / 75% (wet)** of days; dry yield **1836** vs SAC **4205** kg/ha, water **85** vs **484** mm.

**Root cause:** the entropy term was load-bearing for **exploration / replay-buffer coverage**, not just the action-pin. v2.19 replaced it with only a weak, fast-decaying noise (0.20->0.05/100k) + target-policy smoothing (which gives **no** state-action coverage). The actor drifted to the 0 mm corner, the buffer filled with drought transitions, twin-min pessimism kept higher-water actions looking bad, and nothing could recover. Aggravated by `learning_starts=1000` and the **5x asymmetric actor LR**.

## The four fixes in v2.19b

| # | knob | v2.19 | **v2.19b** | why |
|---|---|---|---|---|
| 1 | `learning_starts` | 1,000 | **25,000** | warm-start the critic on random data before the actor exploits it |
| 2 | exploration noise | 0.20->0.05 / 100k | **0.40->0.15 / 150k, floor held** | sustained coverage; TD3's ONLY exploration source |
| 3 | `actor_lr_mult` | 5x | **1x** | stop the actor racing to the boundary ahead of the critic |
| 4 | telemetry | (none) | **LowActionCoverage + CollapseGuard** | log coverage; abort early if collapse recurs |

The `CollapseGuard` aborts the run if the rolling low-action fraction exceeds 60% after 30k steps -> a recurrence costs ~35-40k steps, not a full 250k.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; DRIVE_ROOT='/content/drive/MyDrive/thesis_v219b_td3_runs'; os.makedirs(DRIVE_ROOT,exist_ok=True)
print('Drive mounted:',DRIVE_ROOT)


In [ ]:
# Clone repo + install deps (SB3 pinned 2.6.0). Same stack as the SAC/TD3 runs.
import subprocess, sys, os
WORK='/content'; repo=os.path.join(WORK,'thesis')
if os.path.exists(repo): subprocess.run(['rm','-rf',repo],check=True)
subprocess.run(['git','clone','https://github.com/taratorbati/thesis.git',repo],check=True)
os.chdir(repo); sys.path.insert(0,repo)
subprocess.run(['pip','install','--quiet','stable-baselines3==2.6.0','gymnasium','wandb','pytest'],check=True)
import torch; print(f'PyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY'); print('OK WANDB key.')
except Exception as e:
    try:
        import getpass; k=getpass.getpass('WANDB key (Enter to skip): ').strip()
        if k: os.environ['WANDB_API_KEY']=k; print('OK set.')
        else: print('Skipping WandB.')
    except Exception: print('Skipping WandB.')
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'no GPU')


In [ ]:
# Pre-flight: smoke tests + a TD3 pilot that actually RUNS gradient steps.
# learning_starts=200 < total=1200 so the TD3 update loop executes (critic loss,
# target-policy smoothing, policy_delay, symmetric LR, CollapseGuard wiring) --
# catching runtime bugs before the full run. guard_abort=False so the short
# pilot never false-trips. Then assert the checkpoint is a TD3 actor.
import subprocess, sys
print('Smoke tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_rl_smoke.py','-v','--tb=short']).returncode==0,'SMOKE FAILED'
print('\nFactorized-critic tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_factorized_critic.py','-v','--tb=short']).returncode==0,'CRITIC TESTS FAILED'
print('\nTD3 v2.19b pilot (runs real gradient steps)...')
from src.rl.train_v219b_td3 import train_td3_v219b
m = train_td3_v219b(seed=999, output_dir='/content/pilot', wandb_project=None,
                    total_timesteps=1200, learning_starts=200,
                    explore_decay_steps=300, guard_abort=False)
import glob, zipfile, io, torch
ck = glob.glob('/content/pilot/td3_v219b_seed999/*final*.zip')
if ck:
    with zipfile.ZipFile(ck[0]) as z:
        sd = torch.load(io.BytesIO(z.open('policy.pth').read()), map_location='cpu', weights_only=False)
    assert 'actor.log_std.weight' not in sd, 'BUG: TD3 actor unexpectedly has log_std!'
    assert 'actor.mu_head.weight' in sd, 'BUG: TD3 actor missing mu_head!'
    assert abs(float(sd['actor.obs_norm_marker'].item()) - 2.19) < 0.01, 'BUG: marker != 2.19'
    print('  Checkpoint sanity: deterministic actor (no log_std), mu_head present, marker=2.19. OK')
print('\nOK pre-flight passed. Proceed.')


In [ ]:
# Full 250k TD3 v2.19b training. ~30-55 min A100 / ~2-2.5 h T4.
# The corrected hyperparameters are the function DEFAULTS; shown here explicitly
# for the record. The CollapseGuard will abort early (~35-40k) if collapse recurs.
SEED = 0    # CHANGE per session
from src.rl.train_v219b_td3 import train_td3_v219b
model = train_td3_v219b(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    learning_starts=25_000,                                  # fix 1
    explore_sigma_start=0.40, explore_sigma_end=0.15,        # fix 2
    explore_decay_steps=150_000,                             #   (floor held)
    actor_lr_mult=1.0,                                       # fix 3
    target_policy_noise=0.2, target_noise_clip=0.5, policy_delay=2,
    guard_collapse_frac=0.60, guard_warmup_steps=30_000, guard_abort=True,  # fix 4
)
print('Training complete.')


In [ ]:
import shutil, os, datetime
src=f'/content/thesis/results/rl/td3_v219b_seed{SEED}'
dst=os.path.join(DRIVE_ROOT,f'td3_v219b_seed{SEED}_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree(src,dst,ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to Drive:',dst)


In [ ]:
# Post-training 9-cell eval. runner.py auto-detects the TD3 checkpoint
# (marker 2.19 + no log_std -> TD3.load) and applies RAIN_REF=30 -- NO runner
# change is needed for v2.19b (same architecture/marker as v2.19).
import subprocess, sys, os
model_path=f'/content/thesis/results/rl/td3_v219b_seed{SEED}/best_model/best_model.zip'
final_path=f'/content/thesis/results/rl/td3_v219b_seed{SEED}/td3_v219b_seed{SEED}_final.zip'
print('Evaluating BEST (perfect)...')
r=subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','perfect'],
    capture_output=True,text=True)
print(r.stdout[-1500:])
if r.returncode!=0: print('STDERR:',r.stderr[-2500:])
assert r.returncode==0,'PERFECT EVAL FAILED'
print('\nEvaluating BEST (noisy, robustness)...')
subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
    '--model',model_path,'--scenario','all','--budget','all','--forecast','noisy','--noise-seed','42'])
if os.path.exists(final_path):
    print('\nEvaluating FINAL (250k, perfect)...')
    subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl','--mode','eval',
        '--model',final_path,'--scenario','all','--budget','all','--forecast','perfect'])
print('\nNOTE: outputs under results/runs/<model-name>/.')


In [ ]:
# PRIMARY DIAGNOSTIC + NO-COLLAPSE GATE (the v2.19 regression check).
import pandas as pd, numpy as np, json, glob, os
cands=[d for d in glob.glob('/content/thesis/results/runs/*td3_v219b*') if os.path.isdir(d)]
cands+=[d for d in glob.glob('/content/thesis/results/runs/*') if os.path.isdir(d) and glob.glob(os.path.join(d,'sac_perfect_det_wet*100*seed0.parquet'))]
OUT=next(iter(sorted(set(cands),key=os.path.getmtime,reverse=True)),None)
print('Eval dir:',OUT); assert OUT,'No eval dir found.'

def agg(d, scen):
    ys,ws,wls,x1s,umeans=[],[],[],[],[]
    for b in ['100pct','85pct','70pct']:
        pj=glob.glob(os.path.join(d,f'sac_perfect_det_{scen}*{b}*seed0.json'))
        pq=glob.glob(os.path.join(d,f'sac_perfect_det_{scen}*{b}*seed0.parquet'))
        if not pj or not pq: continue
        m=json.load(open(pj[0]))['final_metrics']; df=pd.read_parquet(pq[0])
        ys.append(m['yield_kg_ha']); ws.append(m.get('water_used_mm'))
        wls.append(m.get('waterlog_days_per_agent')); x1s.append(float(df['x1'].median()))
        umeans.append(float(df['u'].mean()))
    f=lambda a:float(np.mean([v for v in a if v is not None])) if a else float('nan')
    return f(ys),f(ws),f(wls),f(x1s),f(umeans)

y,w,wl,x1,_=agg(OUT,'wet')
dyld,_,_,_,du=agg(OUT,'dry')
print('='*64)
print(f'{"metric":<16s}{"v2.19b":>9s}{"v2.19":>8s}{"v2.18":>8s}{"MPC":>6s}{"target":>9s}')
print(f'{"wet x1 median":<16s}{x1:9.1f}{92:8.0f}{136:8.0f}{130:6.0f}{"< 134":>9s}')
print(f'{"wet waterlog":<16s}{wl:9.1f}{0:8.0f}{37:8.0f}{18:6.0f}{"< 32":>9s}')
print(f'{"wet water mm":<16s}{w:9.0f}{79:8.0f}{336:8.0f}{309:6.0f}{"< 330":>9s}')
print(f'{"wet yield":<16s}{y:9.0f}{0:8s}{3687:8.0f}{3752:6.0f}{"(info)":>9s}'.replace(' 0 ','--'))
print('-'*64)
print('NO-COLLAPSE GATE (v2.19 collapsed here):')
print(f'  dry-year u_mean = {du:.2f} mm/day   (v2.19: 0.92 COLLAPSED; SAC v2.18: 5.20; want > ~4)')
print(f'  dry-year yield  = {dyld:.0f} kg/ha   (v2.19: 1836 COLLAPSED; SAC v2.18: 4205)')
print(f"  {'PASS - no collapse' if du>4.0 else 'FAIL - still collapsing!'}")
print('\nPRIMARY:')
print(f'  wet x1 < 134      : {"PASS" if x1<134 else "FAIL"} ({x1:.1f})')
print(f'  wet waterlog < 32 : {"PASS" if wl<32 else "FAIL"} ({wl:.1f})')


In [ ]:
# STABILITY + COLLAPSE-GUARD DIAGNOSTIC.
import os, glob
run_dir=f'/content/thesis/results/rl/td3_v219b_seed{SEED}'
import pandas as pd
br=os.path.join(run_dir,'bias_ratio_log.csv')
if os.path.exists(br):
    b=pd.read_csv(br); print('--- bias_ratio_log ---'); print(b.to_string(index=False))
    neg=(b['q_pred_mean']<0).any(); mx=b['q_inflation_pct'].abs().max()
    print(f'\n  q_pred_mean ever negative: {neg}  (v2.19 TD3: True, diverged to -145 -- v2.19b should be False)')
    print(f'  max |q_inflation_pct|:     {mx:.1f}%  (target < 80%)')
    print(f'  VERDICT: {"CALIBRATED critic -- exploration fix worked" if (not neg and mx<80) else "still miscalibrated -- investigate"}')
else:
    print('No bias_ratio_log.csv at', br)

# Collapse-guard + coverage logs (new in v2.19b).
cg=os.path.join(run_dir,'collapse_guard_log.csv')
if os.path.exists(cg):
    g=pd.read_csv(cg)
    tripped=int(g['collapsed'].max()) if len(g) else 0
    print(f'\n--- collapse_guard ---  rows={len(g)}  final rolling low-frac={g["frac_low_rolling"].iloc[-1]:.0%}')
    print(f'  guard tripped: {"YES (collapsed)" if tripped else "NO (healthy)"}')
cov=os.path.join(run_dir,'low_action_coverage_log.csv')
if os.path.exists(cov):
    c=pd.read_csv(cov)
    print(f'--- low_action_coverage ---  mean frac_low (last 50)={c["frac_low_action"].tail(50).mean():.0%}')

# critic_loss trajectory (cascade check).
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    runs=glob.glob(os.path.join(run_dir,'tensorboard','*'))
    if runs:
        ea=EventAccumulator(runs[0]); ea.Reload()
        if 'train/critic_loss' in ea.Tags()['scalars']:
            cl=ea.Scalars('train/critic_loss'); mxl=max(e.value for e in cl)
            print(f'\n  max critic_loss = {mxl:.2f}  (STABLE if < 100; v2.7 cascade hit 6.9e12)')
except Exception as e:
    print('tensorboard read skipped:', e)


In [ ]:
# Resume from checkpoint. Fill in the Drive run dir from the archive cell.
# SEED=0; STEP=100_000
# CKPT=f'/content/drive/MyDrive/thesis_v219b_td3_runs/<run-dir>/td3_v219b_seed{SEED}/checkpoints/td3_v219b_seed{SEED}_{STEP}_steps.zip'
# from src.rl.train_v219b_td3 import AsymmetricLRTD3
# from src.rl.networks_td3 import TD3VDNPolicy
# model=AsymmetricLRTD3.load(CKPT, custom_objects={'policy_class':TD3VDNPolicy})
# # model.learn(total_timesteps=..., reset_num_timesteps=False)
# # NOTE: action_noise not restored; recreate NormalActionNoise + ExplorationNoiseDecayCallback
# #       (sigma_start=0.40, sigma_end=0.15, decay_steps=150_000) + CollapseGuardCallback.
